In [4]:
import random, math, heapq
from collections import deque

def make_graph(V, edges):
    inf = float("inf")
    graph = [[inf] * V for _ in range(V)]
    for u, v, w in edges:
        graph[u][v] = w
    return graph

def mst_weight(graph):
    n = len(graph)
    inf = float("inf")
    visited = [False]*n
    edges = [(0, 0)]
    total = 0
    cnt = 0
    
    while edges and cnt < n:
        w, u = heapq.heappop(edges)
        if visited[u]:
            continue
        visited[u] = True
        total += w
        cnt += 1
        for v in range(n):
            if graph[u][v] != inf or graph[v][u] != inf:
                w2 = min(graph[u][v], graph[v][u])
                heapq.heappush(edges, (w2, v))
    return (total, cnt == n)

def path_cost(path, graph):
    cost = 0
    n = len(path)
    for i in range(n-1):
        if graph[path[i]][path[i+1]] == float("inf"):
            return float("inf")
        cost += graph[path[i]][path[i+1]]
    if graph[path[-1]][path[0]] == float("inf"):
        return float("inf")
    cost += graph[path[-1]][path[0]]
    return cost

def fitness(path, graph):
    cost = path_cost(path, graph)
    return -cost if cost != float("inf") else -1e9

def hill_climb(graph, restarts=20, max_iters=2000):
    n = len(graph)
    best = None
    best_fit = -1e9
    
    for _ in range(restarts):
        path = list(range(n))
        random.shuffle(path)
        curr_fit = fitness(path, graph)
        
        for _ in range(max_iters):
            improved = False
            for i in range(n):
                for j in range(i+1, n):
                    new_path = path[:]
                    new_path[i], new_path[j] = new_path[j], new_path[i]
                    new_fit = fitness(new_path, graph)
                    if new_fit > curr_fit:
                        path, curr_fit = new_path, new_fit
                        improved = True
            if not improved:
                break
        if curr_fit > best_fit:
            best, best_fit = path, curr_fit
    
    return best, path_cost(best, graph) != float("inf")

def simulated_annealing(graph, max_iters=5000, temp=100.0, cooling=0.995):
    n = len(graph)
    path = list(range(n))
    random.shuffle(path)
    curr_fit = fitness(path, graph)
    best, best_fit = path[:], curr_fit
    
    for _ in range(max_iters):
        i, j = random.sample(range(n), 2)
        new_path = path[:]
        new_path[i], new_path[j] = new_path[j], new_path[i]
        new_fit = fitness(new_path, graph)
        delta = new_fit - curr_fit
        if delta > 0 or random.random() < math.exp(delta/temp):
            path, curr_fit = new_path, new_fit
        if curr_fit > best_fit:
            best, best_fit = path[:], curr_fit
        temp *= cooling
    
    return best, path_cost(best, graph) != float("inf")

def tabu_search(graph, max_iters=3000, tabu_size=100):
    n = len(graph)
    path = list(range(n))
    random.shuffle(path)
    best, best_fit = path[:], fitness(path, graph)
    tabu = deque(maxlen=tabu_size)
    
    for _ in range(max_iters):
        neighbors = []
        for _ in range(30):
            i, j = random.sample(range(n), 2)
            new_path = path[:]
            new_path[i], new_path[j] = new_path[j], new_path[i]
            if tuple(new_path) not in tabu:
                neighbors.append(new_path)
        
        if not neighbors:
            continue
        
        neighbors.sort(key=lambda p: fitness(p, graph), reverse=True)
        path = neighbors[0]
        tabu.append(tuple(path))
        fit = fitness(path, graph)
        if fit > best_fit:
            best, best_fit = path[:], fit
    
    return best, path_cost(best, graph) != float("inf")

def crossover(p1, p2):
    n = len(p1)
    cut = random.randint(1, n-2)
    child = p1[:cut] + [x for x in p2 if x not in p1[:cut]]
    return child

def mutate(path, rate=0.1):
    new_path = path[:]
    if random.random() < rate:
        i, j = random.sample(range(len(path)), 2)
        new_path[i], new_path[j] = new_path[j], new_path[i]
    return new_path

def genetic_algorithm(graph, pop_size=50, generations=500):
    n = len(graph)
    population = [random.sample(range(n), n) for _ in range(pop_size)]
    
    for _ in range(generations):
        scored = [(fitness(p, graph), p) for p in population]
        scored.sort(reverse=True)
        if path_cost(scored[0][1], graph) != float("inf"):
            return scored[0][1], True
        new_pop = [p for (_, p) in scored[:pop_size//2]]
        while len(new_pop) < pop_size:
            p1, p2 = random.sample(new_pop, 2)
            child = crossover(p1, p2)
            child = mutate(child)
            new_pop.append(child)
        population = new_pop
    
    best = max(population, key=lambda p: fitness(p, graph))
    return best, path_cost(best, graph) != float("inf")

def ham_cycle_solver(V, edges, method="anneal"):
    graph = make_graph(V, edges)
    mst_w, connected = mst_weight(graph)
    if not connected:
        return None, False, "graph disconnected"
    
    if method == "hill":
        return hill_climb(graph)
    elif method == "anneal":
        return simulated_annealing(graph)
    elif method == "tabu":
        return tabu_search(graph)
    elif method == "ga":
        return genetic_algorithm(graph)
    else:
        raise ValueError("unknown method")

In [5]:
V = 5
edges = [(0,1,2),(1,2,2),(2,3,2),(3,4,2),(4,0,2),(0,2,5),(1,3,5),(2,4,5)]

path, found = ham_cycle_solver(V, edges, method="anneal")
if found:
    print("ham cycle found", path, "cost =", path_cost(path, make_graph(V, edges)))
else:
    print("ham cycle not found")

ham cycle found [4, 0, 1, 2, 3] cost = 10
